# Usage Notes

This notebook shows practical examples for working with the data from A multiresolution weather dataset for the Southwestern South Atlantic (2017-2018) **xarray**:

- Calculating **wind speed** from wind components
- Retrieving **wind components at different heights**
- Selecting the **nearest grid-point value**
- Saving data to **csv file**

## libraries

In [ ]:
import numpy as np
import xarray as xr

import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.ticker as ticker
from matplotlib.colors import ListedColormap, BoundaryNorm, LinearSegmentedColormap

import cartopy.crs as ccrs
import cartopy.feature as cfeature

## data loading

In [ ]:
# wrf
data_20170206_D01 = xr.open_dataset('/total/dataset_total/dados/D01/2017/data_D01_20170206.nc')
data_20170206_D02 = xr.open_dataset('/total/dataset_total/dados/D02/2017/data_D02_20170206.nc')
data_20170206_D03 = xr.open_dataset('/total/dataset_total/dados/D03/2017/data_D03_20170206.nc')

# wrf_air_density
air_density_D01 = xr.open_dataset('/total/dataset_total/rho_outputs/air_density_d01_2017_2018_stats.nc')
air_density_D02 = xr.open_dataset('/total/dataset_total/rho_outputs/air_density_d02_2017_2018_stats.nc')
air_density_D03 = xr.open_dataset('/total/dataset_total/rho_outputs/air_density_d03_2017_2018_stats.nc')

# SAR
SAR_1000m = xr.open_dataset('../data_description_article/0_data/d0sar/data_SAR_1000m.nc')

## WRF Data exploration

### Data for February 6, 2018, D01 grid

In [ ]:
data_20170206_D01

#### Terrain height above sea level (HGT)

In [ ]:
# positions and heights
lon = data_20170206_D01["XLONG"]
lat = data_20170206_D01["XLAT"]
hgt = data_20170206_D01["HGT"]

# plot
plt.figure(figsize=(7, 6))
ax = plt.axes(projection=ccrs.PlateCarree())

im = ax.pcolormesh(
    lon, lat, hgt,
    cmap="terrain",
    shading="auto"
)

ax.coastlines()
ax.set_title("D01 – HGT (m)")

plt.colorbar(im, ax=ax, label="HGT (m)")
plt.tight_layout()
plt.show()

# close data file to save memory
data_20170206_D01.close()

#### Land Use Index

In [ ]:
lon = data_20170206_D02["XLONG"]
lat = data_20170206_D02["XLAT"]
lu = data_20170206_D02["LU_INDEX"]


plt.figure(figsize=(7, 6))
ax = plt.axes(projection=ccrs.PlateCarree())

im = ax.pcolormesh(
    lon, lat, lu,
    cmap="tab20",      # categorical-friendly colormap
    shading="auto"
)

ax.coastlines()
ax.set_title("D01 – LU_INDEX")

plt.colorbar(im, ax=ax, label="LU_INDEX")
plt.tight_layout()
plt.show()


### Calculating wind speed at 10 m height

In [ ]:
# read wind components (all times)
u10 = data_20170206_D01["U10"]   # (Time, south_north, west_east)
v10 = data_20170206_D01["V10"]

# calculate wind speed for ALL times (m/s)
wspd = np.sqrt(u10**2 + v10**2)

#### plotting wind speed at 10 m height

In [ ]:
# select time index
tidx = 0  # change this to any Time index you want

ws_t = wspd.isel(Time=tidx)

lon = data_20170206_D01["XLONG"]
lat = data_20170206_D01["XLAT"]

# extract time string
time_str = (
    data_20170206_D01["Times"]
    .isel(Time=tidx)
    .values
)

# plot
plt.figure(figsize=(7, 6))
ax = plt.axes(projection=ccrs.PlateCarree())

im = ax.pcolormesh(
    lon,
    lat,
    ws_t,
    cmap="jet",
    shading="auto"
)

ax.coastlines(resolution="10m", linewidth=0.8)
ax.set_title(f"D01 – Wind speed at 10 m\n{time_str}")

plt.colorbar(im, ax=ax, label="Wind speed (m/s)")
plt.tight_layout()
plt.show()

#### Ploting U wind component

In [ ]:
# --- select time and vertical level ---
time_index = 0
vertical_level = 2  # bottom_top index

# --- select U at fixed time/height (no unstaggering) ---
u = data_20170206_D01["U"].isel(Time=time_index, bottom_top=vertical_level)

# --- lon/lat on U grid ---
lon_u = data_20170206_D01["XLONG_U"]
lat_u = data_20170206_D01["XLAT_U"]

# --- extract time string ---
time_str = (
    data_20170206_D01["Times"]
    .isel(Time=time_index)
    .values
    .astype(str)
    .item()
)

# --- plot ---
fig = plt.figure(figsize=(7, 6))
ax = plt.axes(projection=ccrs.PlateCarree())

im = ax.pcolormesh(
    lon_u, lat_u, u,
    cmap="RdBu_r",
    shading="auto",
    transform=ccrs.PlateCarree(),
)

ax.coastlines(resolution="10m", linewidth=0.8)
ax.set_title(f"D01 – U (bottom_top={vertical_level})\n{time_str}")

plt.colorbar(im, ax=ax, label="U wind component (m/s)")
plt.tight_layout()
plt.show()

### Exporting data to csv

In [ ]:
data_20170206_D03 

In [ ]:
import numpy as np
import pandas as pd

# ============================================================
# USER INPUT (set these first)
# ============================================================
lat0 = -27.50
lon0 = -48.30

ds = data_20170206_D03 

out_csv = "example_u10_v10_timeseries.csv"

# 1) Get 2D lat/lon (mass grid) and drop optional Time dim
lat_da = ds["XLAT"]
lon_da = ds["XLONG"]

if "Time" in lat_da.dims:
    lat_da = lat_da.isel(Time=0)
    lon_da = lon_da.isel(Time=0)

lat2d = lat_da.values
lon2d = lon_da.values

# 2) Warn if outside grid bounding box (simple check)
lat_min = float(np.nanmin(lat2d))
lat_max = float(np.nanmax(lat2d))
lon_min = float(np.nanmin(lon2d))
lon_max = float(np.nanmax(lon2d))

outside = (lat0 < lat_min) or (lat0 > lat_max) or (lon0 < lon_min) or (lon0 > lon_max)
if outside:
    print(
        f"⚠️ Point ({lat0:.5f}, {lon0:.5f}) is outside grid bounding box:\n"
        f"    lat range: [{lat_min:.5f}, {lat_max:.5f}]\n"
        f"    lon range: [{lon_min:.5f}, {lon_max:.5f}]"
    )

# 3) Find closest grid point using haversine distance (km)
R = 6371.0  # km

lat_rad = np.deg2rad(lat2d)
lon_rad = np.deg2rad(lon2d)
lat0r = np.deg2rad(lat0)
lon0r = np.deg2rad(lon0)

dlat = lat_rad - lat0r
dlon = lon_rad - lon0r

a = np.sin(dlat / 2.0) ** 2 + np.cos(lat0r) * np.cos(lat_rad) * np.sin(dlon / 2.0) ** 2
c = 2.0 * np.arctan2(np.sqrt(a), np.sqrt(1.0 - a))
dist_km = R * c

flat_idx = int(np.nanargmin(dist_km))
j, i = np.unravel_index(flat_idx, dist_km.shape)
d_km = float(dist_km[j, i])

closest_lat = float(lat2d[j, i])
closest_lon = float(lon2d[j, i])

print("\n📍 Closest grid point (mass grid: XLAT/XLONG)")
print(f"Target lat/lon : ({lat0:.5f}, {lon0:.5f})")
print(f"Closest ij     : ({j}, {i})")
print(f"Closest lat/lon: ({closest_lat:.5f}, {closest_lon:.5f})")
print(f"Distance       : {d_km:.2f} km")

# 4) Extract U10/V10 for ALL times at that grid point
u10_ts = ds["U10"].isel(south_north=j, west_east=i).values
v10_ts = ds["V10"].isel(south_north=j, west_east=i).values

nT = ds.sizes["Time"]
time_strs = [
    ds["Times"].isel(Time=t).values.astype(str).item()
    for t in range(nT)
]

# 5) Save to CSV
df = pd.DataFrame(
    {
        "time_str": time_strs,
        "U10_mps": u10_ts.astype(float),
        "V10_mps": v10_ts.astype(float),
        "target_lat": lat0,
        "target_lon": lon0,
        "closest_lat": closest_lat,
        "closest_lon": closest_lon,
        "distance_km": d_km,
    }
)

df.to_csv(out_csv, index=False)
print(f"\n✅ Saved {len(df)} rows to: {out_csv}")

### WRF Air density

In [ ]:
air_density_D01

### Air density aggregations

In [ ]:
DATASETS = {
    "D01": air_density_D01,
    "D02": air_density_D02,
    "D03": air_density_D03,
}

CMAP_MEAN = "jet"
CMAP_STD = "Greys"

P_LO, P_HI = 1, 99
COAST_RES = "10m"
COAST_LW = 0.7

# plot for all domains
for domain, ds in DATASETS.items():

    rho = ds["air_density"]
    lat = ds["XLAT"]
    lon = ds["XLONG"]

    # safety: drop Time if present
    if "Time" in lat.dims:
        lat = lat.isel(Time=0, drop=True)
    if "Time" in lon.dims:
        lon = lon.isel(Time=0, drop=True)

    rho_mean = rho.sel(aggregation="mean")
    rho_std  = rho.sel(aggregation="std")

    n_levels = rho_mean.sizes["bottom_top"]

    mean_vals = rho_mean.values
    std_vals = rho_std.values

    vmin_mean, vmax_mean = np.nanpercentile(mean_vals, [P_LO, P_HI])
    vmin_std,  vmax_std  = np.nanpercentile(std_vals,  [P_LO, P_HI])

    proj = ccrs.PlateCarree()
    fig, axes = plt.subplots(
        nrows=2,
        ncols=n_levels,
        figsize=(4.2 * n_levels, 7.0),
        subplot_kw={"projection": proj},
    )

    im_mean = None
    im_std = None

    for k in range(n_levels):
        # --- mean ---
        ax = axes[0, k]
        im_mean = ax.pcolormesh(
            lon, lat, rho_mean.isel(bottom_top=k),
            transform=proj, shading="auto",
            cmap=CMAP_MEAN, vmin=vmin_mean, vmax=vmax_mean,
        )
        ax.coastlines(resolution=COAST_RES, linewidth=COAST_LW)
        ax.set_title(f"Level {k+1}", fontsize=10)

        # --- std ---
        ax = axes[1, k]
        im_std = ax.pcolormesh(
            lon, lat, rho_std.isel(bottom_top=k),
            transform=proj, shading="auto",
            cmap=CMAP_STD, vmin=vmin_std, vmax=vmax_std,
        )
        ax.coastlines(resolution=COAST_RES, linewidth=COAST_LW)

    # layout
    fig.subplots_adjust(
        left=0.08, right=0.88, top=0.92, bottom=0.08,
        wspace=0.06, hspace=0.02,
    )

    # row labels
    axes[0, 0].text(
        -0.06, 0.5, "Spatial mean",
        transform=axes[0, 0].transAxes,
        rotation=90, va="center", ha="right",
        fontsize=12, fontweight="bold",
    )
    axes[1, 0].text(
        -0.06, 0.5, "Spatial standard deviation",
        transform=axes[1, 0].transAxes,
        rotation=90, va="center", ha="right",
        fontsize=12, fontweight="bold",
    )

    # colorbars
    cax_mean = fig.add_axes([0.90, 0.53, 0.015, 0.36])
    cb_mean = fig.colorbar(im_mean, cax=cax_mean)
    cb_mean.set_label("air density (kg m$^{-3}$)")

    cax_std = fig.add_axes([0.90, 0.10, 0.015, 0.36])
    cb_std = fig.colorbar(im_std, cax=cax_std)
    cb_std.set_label("air density (kg m$^{-3}$)")

    fig.suptitle(f"{domain} — air density (mean / std)", y=0.97)
    plt.show()


## SAR data

In [ ]:
SAR_1000m

### Spatial average per acquisition

In [ ]:
# convert time to datetime
time = pd.to_datetime(
    [x.decode("utf-8") for x in SAR_1000m["Times"].values],
    format="%Y-%m-%d_%H:%M:%S",
    errors="raise",
)

# spatial mean over lat/lon
wspd_mean = SAR_1000m["WIND_SPD"].mean(dim=("south_north", "west_east"))

# Plot
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(time, wspd_mean, "o-", label="SAR / CMOD5 1000 m resolution (spatial mean per acquisition)")

# --- x-axis formatting: MMM \n YYYY ---
def fmt_date(x, pos=None):
    return mdates.num2date(x).strftime("%b\n%Y").title()

# styling
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=2))
ax.xaxis.set_major_formatter(ticker.FuncFormatter(fmt_date))
ax.tick_params(axis="both", which="major", labelsize=14)
ax.set_ylabel("Wind Speed (m/s)", fontsize=17)
ax.legend(fontsize=14, loc="upper right")
plt.tight_layout(pad=2.0)

plt.show()
